# Notebook 3: Model Architecture

**Purpose:** This notebook defines and inspects the AtrionNet v4.0 architecture.
It explains each component of the model and verifies that the model works correctly
before any training begins.

## Architecture: Attentional-Hybrid (CNN + BiLSTM + SE-Attention)

The model is built from three main stages:

| Stage | Component | Purpose |
|---|---|---|
| **Encoder** | Attentional Inception Blocks | Extract multi-scale ECG shapes (P, QRS, T) |
| **Bridge** | Bidirectional LSTM | Model the temporal rhythm of the heartbeat |
| **Decoder** | Upsampling + Skip Connections | Reconstruct per-sample predictions |
| **Output Heads** | Heatmap, Width, Mask | Detect, localize, and quantify P-waves |

**Key Research Innovation:** The Squeeze-and-Excitation (SE) Attention gate suppresses the strong
QRS complex so the model can focus on the weaker, overlapping P-waves.

## Step 1: Setup

In [ ]:
import sys
import os
import torch
from pathlib import Path

PROJECT_ROOT = str(Path(os.getcwd()).parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## Step 2: Import Model Classes

We import both models:
- `AtrionNetHybrid`: The main research model (CNN + BiLSTM + Attention)
- `AtrionNetBaseline`: A simple CNN used only for the ablation study comparison

In [ ]:
from src.modeling.atrion_net import AtrionNetHybrid, AtrionNetBaseline

print("AtrionNetHybrid and AtrionNetBaseline imported successfully.")

## Step 3: Instantiate the Hybrid Model

We create the model with 12 input channels (one per ECG lead).

In [ ]:
model = AtrionNetHybrid(in_channels=12).to(DEVICE)

# Count total and trainable parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Step 4: Print the Model Structure

This shows every layer in the model, which is useful for explaining the architecture in your thesis.

In [ ]:
print(model)

## Step 5: Verify Forward Pass with Dummy Data

We pass a dummy batch through the model to verify that all tensor shapes are correct.
A batch of 2 records, 12 leads, 5000 samples is used.

In [ ]:
import torch

# Dummy input: batch=2, leads=12, samples=5000
dummy_input = torch.randn(2, 12, 5000).to(DEVICE)

with torch.no_grad():
    output = model(dummy_input)

print("Forward pass successful. Output shapes:")
for key, val in output.items():
    print(f"  {key:10s}: {val.shape}")

## Step 6: Explain Each Output Head

The model produces three outputs for each input ECG record:

- **Heatmap** `[Batch, 1, 5000]`: A probability score at each time step. High values indicate the center (peak) of a P-wave.
- **Width** `[Batch, 1, 5000]`: The predicted normalized width of the P-wave at each detected center.
- **Mask** `[Batch, 1, 5000]`: A binary-like segmentation mask — 1 where a P-wave exists, 0 elsewhere.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot the raw model output for one dummy record to show what the heads look like
heatmap = output['heatmap'][0, 0].cpu().numpy()
mask    = output['mask'][0, 0].cpu().numpy()

fig, axes = plt.subplots(2, 1, figsize=(15, 6))

axes[0].plot(heatmap, color='crimson')
axes[0].set_title('Heatmap Head Output (Untrained) — P-wave Center Probability')
axes[0].set_ylabel('Score')

axes[1].plot(mask, color='steelblue')
axes[1].set_title('Mask Head Output (Untrained) — P-wave Segmentation')
axes[1].set_ylabel('Score')
axes[1].set_xlabel('Sample Index')

plt.tight_layout()
plt.show()

print("Note: Outputs look random here because the model is not yet trained.")

---
**Architecture verified. Proceed to `04_training/model_training.ipynb`.**